# VisionAid · YOLO11n: train and evaluate on dataset v4

One of five notebooks that compare detectors under **identical conditions**: the same data, the same split, the same training budget, the same evaluator. Only the model changes. Each notebook:

1. downloads Roboflow **version2-nvt5l v4**;
2. **re-splits it by scene**, exactly as the dataset check notebook does (videos, photo sets and look-alike scenes stay in one split; product, stock and web wet floor signs go to train only; images with an unlabelled hazard are dropped; background images are capped at 10 %). The split is deterministic, so all five notebooks get the same split; the fingerprint `manifest_md5` in the results proves it;
3. trains **YOLO11n** for up to 100 epochs with early stopping;
4. scores it with one **shared evaluator** (COCO mAP with pycocotools at confidence 0.001, alert threshold chosen on validation by best F1, latency at batch 1);
5. writes `visionaid_runs/results/yolo11n_v4_seed0.json` for the comparison notebook, and copies the best weights to `visionaid_runs/weights/`.

**Before you run:** Settings ▸ Accelerator **GPU T4**, Settings ▸ Internet **on**, and a Kaggle secret named `ROBOFLOW_API_KEY` (Add-ons ▸ Secrets) holding a **new** Roboflow key. Then **Save Version ▸ Save & Run All**, so it runs in the background and its output can be downloaded or attached later.

In [1]:
# ✍️ Settings: only MODEL changes between the five notebooks
import os
# ROBOFLOW_API_KEY is read from your environment or Kaggle secrets below.
MODEL = "yolo11n"              # YOLO11n
ARCH, FAMILY, WEIGHTS = "YOLO11", "ultralytics", 'yolo11n.pt'

# data: the same for all five notebooks (keep identical, or the comparison is not fair)
WORKSPACE, PROJECT, VERSION = "potatos-workspace-kequm", "version2-nvt5l", 4
SEED = 0
GROUP_OVERRIDES = {"IMG_0285_MOV": "spot-A", "IMG_0286_MOV": "spot-A", "IMG_0287_MOV": "spot-A"}
EASY_SOURCES = ("wet-floor-sign", "wet floor sign", "2023-05-30", "istockphoto", "adobestock",
                "gettyimages", "shutterstock", "images")       # product, stock, web and tabletop sets: train only
SPLIT_FRAC = {"valid": 0.10, "test": 0.10}
PRIORITY_CLASSES = ("Wet Floor Sign",)
PRIORITY_FRAC = {"valid": 0.15, "test": 0.15}
DROP_UNLABELLED_HINTS = ("step", "escalator", "stair")
HINTS = {"step": ("step", "stair"), "stair": ("stair",), "escalator": ("escalator",),
         "wet": ("wet", "sign"), "pillar": ("pillar",)}
BG_KEEP = 0.10            # background images (no boxes) kept at about 10 % of each split
DUP_HAMMING, TRIES = 6, 200

# training: the same for all five notebooks
EPOCHS, PATIENCE, BATCH, IMGSZ, WORKERS = 100, 25, 16, 640, 4
SAFETY_CLASSES = ("Stair", "Escalator", "Wet Floor Sign")   # the classes where a miss is dangerous

WORK = "/kaggle/working" if __import__("os").path.isdir("/kaggle/working") else "."
DATA_ROOT = None                                   # a folder with the Roboflow YOLO download; None = download
DATA_DIR = f"{WORK}/visionaid_v{VERSION}_resplit"
RUN_NAME = f"{MODEL}_v{VERSION}_seed{SEED}"
RESULTS_DIR = f"{WORK}/visionaid_runs/results"

In [2]:
# ▶ Install, import, download (about 2 minutes)
import os, re, glob, json, time, random, shutil, hashlib, subprocess, sys, importlib.util, collections
need = ["roboflow", "pycocotools"] + (["ultralytics==8.4.183"] if FAMILY == "ultralytics" else ["rfdetr[train,loggers]==1.11.2"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)   # pinned versions, same in every run
import numpy as np, pandas as pd, yaml, torch
from PIL import Image
SPLITS = ("train", "valid", "test")
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (training will be very slow)")

if DATA_ROOT is None:
    key = os.environ.get("ROBOFLOW_API_KEY")
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        except Exception:
            raise RuntimeError("Add your Roboflow API key as a Kaggle secret named ROBOFLOW_API_KEY (Add-ons ▸ Secrets).")
    from roboflow import Roboflow
    DATA_ROOT = (Roboflow(api_key=key).workspace(WORKSPACE).project(PROJECT).version(VERSION)
                 .download("yolov8", location=f"{WORK}/{PROJECT}-v{VERSION}").location)
print("download:", DATA_ROOT)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


device: Tesla T4
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/version2-nvt5l-v4 in yolov8:: 100%|██████████| 25011/25011 [00:02<00:00, 10336.91it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
download: /kaggle/working/version2-nvt5l-v4


### Data: download and re-split
The same code and settings in all five notebooks, so they all train and test on the same images. Compare the printed **split fingerprint** across the five notebooks: it must be identical.

In [3]:
# ▶ Re-split by scene: the same rules as the dataset check notebook
CAMERA = re.compile(r"^(img|pxl|dsc|dscn|dcim|photo|screenshot|whatsapp|signal|mvimg)[\w\- ]*$", re.I)
VIDEO = re.compile(r"^(.*?_(?:mp4|mov|avi|mkv|m4v|webm|3gp))-\d+$", re.I)
DATE = re.compile(r"^(\d{4}-\d{2}-\d{2}|\d{8})(?=[\s_\-]|$)")

def source_group(stem):
    s = stem.split(".rf.")[0]
    s = re.sub(r"_(jpe?g|png|webp|heic|bmp)$", "", s, flags=re.I)
    for rx in (VIDEO, DATE):
        m = rx.match(s)
        if m:
            return GROUP_OVERRIDES.get(m.group(1), m.group(1))
    s = re.sub(r"-\d+x\d+$", "", s)
    if not CAMERA.match(s):
        for _ in range(2):
            t = re.sub(r"[\s_\-]*\(?\d+\)?[\s_\-]*$", "", s)
            if not t or t == s:
                break
            s = t
    return GROUP_OVERRIDES.get(s, s)

def load_names(root):
    names = yaml.safe_load(open(os.path.join(root, "data.yaml")))["names"]
    return [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)

def read_label(path):
    out = []
    if os.path.exists(path):
        for line in open(path):
            v = line.split()
            if len(v) >= 5:
                vals = list(map(float, v[1:]))
                if len(vals) == 4:
                    cx, cy, w, h = vals; box = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
                else:
                    xs, ys = vals[0::2], vals[1::2]; box = (min(xs), min(ys), max(xs), max(ys))
                out.append((int(float(v[0])), box))
    return out

def build_index(root):
    rows = []
    for s in SPLITS:
        for img in sorted(glob.glob(os.path.join(root, s, "images", "*"))):
            if img.lower().endswith(IMG_EXT):
                stem = os.path.splitext(os.path.basename(img))[0]
                lbl = os.path.join(root, s, "labels", stem + ".txt")
                rows.append(dict(split=s, image=img, label_path=lbl, stem=stem, group=source_group(stem), labels=read_label(lbl)))
    df = pd.DataFrame(rows)
    df["classes"] = df.labels.map(lambda L: sorted({c for c, _ in L}))
    df["easy"] = df.stem.str.lower().map(lambda s: s.startswith(EASY_SOURCES))
    return df

norm = lambda n: re.sub(r"[\s_\-]+", " ", n.lower()).strip()
NAMES = load_names(DATA_ROOT)
WET_IDS = [i for i, n in enumerate(NAMES) if "wet" in norm(n)]
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def dhash(path, size=8):
    a = np.asarray(Image.open(path).convert("L").resize((size + 1, size)), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).ravel()

def close_pairs(paths, thr=DUP_HAMMING, chunk=400):
    if not len(paths):
        return np.zeros((0, 2), int)
    P = np.packbits(np.stack([dhash(p) for p in paths]), axis=1)
    out = []
    for a in range(0, len(P), chunk):
        d = POP[P[a:a + chunk, None, :] ^ P[None, :, :]].sum(-1)
        i, j = np.nonzero(d <= thr)
        keep = (i + a) < j
        out.append(np.stack([i[keep] + a, j[keep]], 1))
    return np.concatenate(out)

def near_copies(df):
    sel = df[df.classes.map(lambda c: any(i in WET_IDS for i in c))]
    tr = sel[sel.split == "train"]
    if not len(tr):
        return set()
    H = np.stack([dhash(p) for p in tr.image])
    return {r.image for _, r in sel[sel.split != "train"].iterrows() if (H != dhash(r.image)).sum(1).min() <= DUP_HAMMING}

def hint_drop(df):
    drop = set()
    for _, r in df.iterrows():
        name, labelled = r.stem.lower(), [norm(NAMES[c]) for c in r.classes]
        for key in DROP_UNLABELLED_HINTS:
            if key in name and not any(o in lab for o in HINTS[key] for lab in labelled):
                drop.add(r.image)
    return drop

DF = build_index(DATA_ROOT)
WORKDF = DF[~DF.image.isin(hint_drop(DF))].copy()                       # 1. unlabelled hazards out
sel = WORKDF[WORKDF.classes.map(lambda c: any(i in WET_IDS for i in c))]
parent = {g: g for g in WORKDF.group.unique()}
def find(g):
    while parent[g] != g:
        parent[g] = parent[parent[g]]; g = parent[g]
    return g
gs = list(sel.group)
for i, j in close_pairs(list(sel.image)):                               # 2. look-alike sources = one scene
    a, b = find(gs[i]), find(gs[j])
    if a != b:
        parent[b] = a
WORKDF["scene"] = WORKDF.group.map(find)

gsize = WORKDF.groupby("scene").size()                                  # 3. assign whole scenes
gbox = {g: collections.Counter(c for L in part.labels for c, _ in L) for g, part in WORKDF.groupby("scene")}
cls_total = collections.Counter(c for L in WORKDF.labels for c, _ in L)
forced = set(WORKDF[WORKDF.easy].scene)
free = sorted(set(gsize.index) - forced)
pri = [NAMES.index(n) for n in PRIORITY_CLASSES if n in NAMES] or WET_IDS
free_total = {c: sum(gbox[g][c] for g in free) for c in cls_total}
cap = {s: SPLIT_FRAC[s] * len(WORKDF) * 1.05 for s in ("valid", "test")}
pri0 = [g for g in free if any(gbox[g][c] for c in pri)]
rest0 = [g for g in free if g not in set(pri0)]
rng, best = random.Random(SEED), None
for t in range(TRIES):
    pg, rg = pri0[:], rest0[:]; rng.shuffle(pg); rng.shuffle(rg)
    assign = {g: "train" for g in forced}
    filled, pfill = {"valid": 0, "test": 0}, {"valid": 0, "test": 0}
    pri_free = sum(free_total[c] for c in pri)
    for g in pg:
        open_ = [s for s in ("valid", "test") if pfill[s] < PRIORITY_FRAC[s] * pri_free and filled[s] + gsize[g] <= cap[s]]
        s = min(open_, key=lambda s: pfill[s] / PRIORITY_FRAC[s]) if open_ else "train"
        assign[g] = s
        if s != "train":
            filled[s] += gsize[g]; pfill[s] += sum(gbox[g][c] for c in pri)
    for g in rg:
        s = next((s for s in ("valid", "test") if filled[s] + gsize[g] <= cap[s]), "train")
        assign[g] = s
        if s != "train":
            filled[s] += gsize[g]
    share = {s: collections.Counter() for s in SPLITS}
    for g, s in assign.items():
        share[s].update(gbox[g])
    score = 0.0
    for c in cls_total:
        for s in ("valid", "test"):
            score += (2 * abs(share[s][c] / max(free_total[c], 1) - PRIORITY_FRAC[s]) if c in pri
                      else abs(share[s][c] / cls_total[c] - SPLIT_FRAC[s])) + 10 * (share[s][c] == 0)
    if best is None or score < best[0]:
        best = (score, dict(assign))
NEW = WORKDF.copy()
NEW["split"] = NEW.scene.map(best[1])
NEW = NEW[~NEW.image.isin(near_copies(NEW))].copy()                      # 4. near-copies out of valid/test
rng_bg = random.Random(SEED)
for s in SPLITS:                                                        # 5. background capped at ~10 %
    part = NEW[NEW.split == s]
    bg = part[part.labels.map(len) == 0]
    max_bg = int(BG_KEEP * (len(part) - len(bg)) / (1 - BG_KEEP))
    if len(bg) > max_bg:
        NEW = NEW.drop(index=rng_bg.sample(list(bg.index), len(bg) - max_bg))

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
for s in SPLITS:
    os.makedirs(f"{DATA_DIR}/{s}/images"); os.makedirs(f"{DATA_DIR}/{s}/labels")
for _, r in NEW.iterrows():
    shutil.copy2(r.image, f"{DATA_DIR}/{r.split}/images/")
    if r.labels:                    # every label as a plain box: polygon outlines become the box around them
        with open(f"{DATA_DIR}/{r.split}/labels/{r.stem}.txt", "w") as f:
            for c, (x1, y1, x2, y2) in r.labels:
                x1, y1, x2, y2 = max(0.0, x1), max(0.0, y1), min(1.0, x2), min(1.0, y2)
                f.write(f"{c} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} {x2 - x1:.6f} {y2 - y1:.6f}\n")
N_POLY = sum(1 for p in NEW.label_path if os.path.exists(p) and any(len(l.split()) > 5 for l in open(p)))
yaml.safe_dump({"path": os.path.abspath(DATA_DIR), "train": "train/images", "val": "valid/images", "test": "test/images",
                "nc": len(NAMES), "names": NAMES}, open(f"{DATA_DIR}/data.yaml", "w"), sort_keys=False)
manifest = NEW[["stem", "scene", "split"]].sort_values("stem")
manifest.to_csv(f"{DATA_DIR}/split_manifest.csv", index=False)
MANIFEST_MD5 = hashlib.md5(manifest.to_csv(index=False).encode()).hexdigest()

SPLIT_IMAGES = {s: int((NEW.split == s).sum()) for s in SPLITS}
scene_split = NEW.groupby("scene").split.nunique()
LEAK = {f"{s}_in_train_frac": float(NEW[NEW.split == s].scene.isin(set(NEW[NEW.split == "train"].scene)).mean()) for s in ("valid", "test")}
assert scene_split.max() == 1 and max(LEAK.values()) == 0, "a scene is in two splits: the re-split went wrong"
counts = pd.DataFrame({s: collections.Counter(NAMES[c] for L in NEW[NEW.split == s].labels for c, _ in L) for s in SPLITS}).reindex(NAMES).fillna(0).astype(int)
print("classes:", NAMES)
print("images:", SPLIT_IMAGES, "| split fingerprint:", MANIFEST_MD5)
print(f"label files with polygon outlines converted to boxes: {N_POLY}")
print("boxes per class:"); print(counts.to_string())

classes: ['Escalator', 'People', 'Pillar', 'Stair', 'Wet Floor Sign']
images: {'train': 8247, 'valid': 925, 'test': 920} | split fingerprint: 49601080aac1f838190341d331785578
label files with polygon outlines converted to boxes: 161
boxes per class:
                train  valid  test
Escalator         944    121   158
People           8374    818   569
Pillar           1156    161   153
Stair            2822    321   288
Wet Floor Sign   1621    118   116


### Train YOLO11n
Up to 100 epochs, early stopping after 25 epochs without improvement on validation, effective batch 16, seed 0. Augmentation is applied during training only, to training images only.

In [4]:
# ▶ Train YOLO11n (about 2-3 hours on a T4)
from ultralytics import YOLO, __version__ as LIB_VERSION
model = YOLO(WEIGHTS)
t0 = time.time()
model.train(data=f"{DATA_DIR}/data.yaml", imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
            seed=SEED, deterministic=True, workers=WORKERS, cos_lr=True, device=0 if DEVICE == "cuda" else "cpu",
            project=os.path.abspath(f"{WORK}/runs"), name=RUN_NAME, exist_ok=True,
            mosaic=1.0, close_mosaic=10, scale=0.5, translate=0.1, fliplr=0.5, flipud=0.0,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4)
TRAIN_SECONDS = time.time() - t0
BEST = str(model.trainer.best)                     # the folder Ultralytics really used
model = YOLO(BEST)
MODEL_INPUT = IMGSZ

def predict(path):
    # every detection at confidence >= 0.001: rows of x1 y1 x2 y2 confidence class, in pixels of the original image
    b = model.predict(path, imgsz=IMGSZ, conf=0.001, iou=0.7, max_det=300, verbose=False,
                      device=0 if DEVICE == "cuda" else "cpu")[0].boxes
    return np.c_[b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()] if len(b) else np.zeros((0, 6))
print(f"trained in {TRAIN_SECONDS / 60:.0f} min, best weights: {BEST}")

New https://pypi.org/project/ultralytics/8.4.185 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/visionaid_v4_resplit/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yol

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      2.74G      1.266      1.352      1.398         20        640: 100% ━━━━━━━━━━━━ 516/516 7.4it/s 1:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 6.6it/s 4.4s
                   all        925       1539      0.324      0.372      0.324       0.17

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      2.74G      1.357      1.506      1.482         27        640: 100% ━━━━━━━━━━━━ 516/516 7.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 6.3it/s 4.6s
                   all        925       1539      0.435      0.351      0.328      0.179

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      2.74G      1.417      1.581      1.536         14        640: 100% ━━━━━━━━━━━━ 516/516 7.9it/s 1:06
                 Class     Images  Instances      Box(P 

### Evaluate with the shared evaluator
COCO mAP from pycocotools on every detection at confidence ≥ 0.001. The alert threshold is the one with the best F1 on **validation**; precision, recall and safety recall on **test** are reported at that threshold. Safety recall is the mean recall over Stair, Escalator and Wet Floor Sign.

In [5]:
# ▶ The shared evaluator: identical in all five notebooks
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import contextlib, io

def gt_and_preds(split):
    rows = NEW[NEW.split == split].sort_values("stem")
    images, anns, dets = [], [], []
    for k, (_, r) in enumerate(rows.iterrows()):
        path = f"{DATA_DIR}/{split}/images/{os.path.basename(r.image)}"
        W, H = Image.open(path).size
        images.append({"id": k, "file_name": os.path.basename(path), "width": W, "height": H})
        for c, (x1, y1, x2, y2) in r.labels:
            bw, bh = (x2 - x1) * W, (y2 - y1) * H
            anns.append({"id": len(anns) + 1, "image_id": k, "category_id": c + 1, "bbox": [x1 * W, y1 * H, bw, bh],
                         "area": bw * bh, "iscrowd": 0})
        dets.append(predict(path))
    gt = {"images": images, "annotations": anns, "categories": [{"id": i + 1, "name": n} for i, n in enumerate(NAMES)]}
    return gt, dets

def coco_map(gt, dets):
    G = COCO(); G.dataset = gt
    with contextlib.redirect_stdout(io.StringIO()):
        G.createIndex()
        res = [{"image_id": k, "category_id": int(d[5]) + 1, "bbox": [float(d[0]), float(d[1]), float(d[2] - d[0]), float(d[3] - d[1])],
                "score": float(d[4])} for k, D in enumerate(dets) for d in D]
        E = COCOeval(G, G.loadRes(res) if res else COCO(), "bbox")
        E.evaluate(); E.accumulate(); E.summarize()
    per = {}
    for i, n in enumerate(NAMES):
        p = E.eval["precision"][0, :, i, 0, -1]                  # IoU 0.5, all areas, 100 detections
        per[n] = float(p[p > -1].mean()) if (p > -1).any() else None
    return float(E.stats[1]), float(E.stats[0]), per

def iou(a, b):
    x1, y1 = np.maximum(a[:, None, 0], b[None, :, 0]), np.maximum(a[:, None, 1], b[None, :, 1])
    x2, y2 = np.minimum(a[:, None, 2], b[None, :, 2]), np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def counts_at(gt, dets, thr):
    # true positives, false positives and missed boxes per class at one alert threshold (IoU 0.5, greedy)
    tp, fp, fn = (collections.Counter() for _ in range(3))
    by_img = collections.defaultdict(list)
    for a in gt["annotations"]:
        x, y, w, h = a["bbox"]; by_img[a["image_id"]].append((a["category_id"] - 1, [x, y, x + w, y + h]))
    for k, D in enumerate(dets):
        D = D[D[:, 4] >= thr] if len(D) else D
        for c in range(len(NAMES)):
            g = np.array([b for cc, b in by_img[k] if cc == c]).reshape(-1, 4)
            d = D[D[:, 5] == c] if len(D) else np.zeros((0, 6))
            d = d[np.argsort(-d[:, 4])]
            used = np.zeros(len(g), bool)
            if len(g) and len(d):
                M = iou(d[:, :4], g)
                for row in M:
                    row = np.where(used, -1, row); j = int(row.argmax())
                    if row[j] >= 0.5:
                        used[j] = True; tp[c] += 1
                    else:
                        fp[c] += 1
            else:
                fp[c] += len(d)
            fn[c] += int((~used).sum())
    return tp, fp, fn

prf = lambda tp, fp, fn: (tp / max(tp + fp, 1), tp / max(tp + fn, 1))

# alert threshold: best F1 on the validation split (the test split never chooses anything)
gt_v, det_v = gt_and_preds("valid")
sweep = []
for thr in np.round(np.arange(0.05, 0.951, 0.05), 2):
    tp, fp, fn = counts_at(gt_v, det_v, thr)
    p, r = prf(sum(tp.values()), sum(fp.values()), sum(fn.values()))
    sweep.append((thr, p, r, 2 * p * r / max(p + r, 1e-9)))
SWEEP = pd.DataFrame(sweep, columns=["threshold", "precision", "recall", "f1"])
CONF = float(SWEEP.loc[SWEEP.f1.idxmax(), "threshold"])
val_map50, val_map, _ = coco_map(gt_v, det_v)

# test: scored once
gt_t, det_t = gt_and_preds("test")
map50, map5095, ap50 = coco_map(gt_t, det_t)
tp, fp, fn = counts_at(gt_t, det_t, CONF)
P, R = prf(sum(tp.values()), sum(fp.values()), sum(fn.values()))
per_class = {}
for i, n in enumerate(NAMES):
    p, r = prf(tp[i], fp[i], fn[i])
    per_class[n] = {"ap50": ap50[n], "recall": r if tp[i] + fn[i] else None, "precision": p, "boxes": tp[i] + fn[i]}
safety = [per_class[n]["recall"] for n in SAFETY_CLASSES if n in per_class and per_class[n]["recall"] is not None]
SAFETY_RECALL = float(np.mean(safety)) if safety else None
print(f"alert threshold (best F1 on validation): {CONF}")
print(f"TEST  mAP50 {map50:.3f} · mAP50-95 {map5095:.3f} · precision {P:.3f} · recall {R:.3f} · safety recall {SAFETY_RECALL:.3f}")
print(pd.DataFrame(per_class).T.round(3).to_string())

alert threshold (best F1 on validation): 0.35
TEST  mAP50 0.658 · mAP50-95 0.405 · precision 0.797 · recall 0.594 · safety recall 0.683
                 ap50  recall  precision  boxes
Escalator       0.948   0.861      0.919  158.0
People          0.729   0.613      0.777  569.0
Pillar          0.196   0.163      0.439  153.0
Stair           0.757   0.670      0.801  288.0
Wet Floor Sign  0.658   0.517      0.968  116.0


In [6]:
# ▶ Latency: batch 1, 10 warm-up runs, then 100 test images, including pre- and post-processing
paths = [f"{DATA_DIR}/test/images/{os.path.basename(p)}" for p in NEW[NEW.split == "test"].sort_values("stem").image][:110]
sync = torch.cuda.synchronize if DEVICE == "cuda" else (lambda: None)
for p in paths[:10]:
    predict(p)
times = []
for p in paths[10:110]:
    sync(); t = time.perf_counter(); predict(p); sync(); times.append(1000 * (time.perf_counter() - t))
LAT = {"p50_ms": float(np.percentile(times, 50)), "p95_ms": float(np.percentile(times, 95)), "fps": float(1000 / np.mean(times))}

def torch_module(obj):
    for path in ("model", "model.model", "model.model.model"):
        o = obj
        for a in path.split("."):
            o = getattr(o, a, None)
        if isinstance(o, torch.nn.Module):
            return o
PARAMS_M = sum(p.numel() for p in torch_module(model).parameters()) / 1e6
WEIGHTS_MB = os.path.getsize(BEST) / 2**20
print(f"latency p50 {LAT['p50_ms']:.1f} ms · p95 {LAT['p95_ms']:.1f} ms · {LAT['fps']:.0f} FPS · {PARAMS_M:.1f} M params · {WEIGHTS_MB:.1f} MB")

latency p50 11.7 ms · p95 12.6 ms · 85 FPS · 2.6 M params · 5.3 MB


In [7]:
# ▶ Save the results file and the weights
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(f"{WORK}/visionaid_runs/weights", exist_ok=True)
shutil.copy2(BEST, f"{WORK}/visionaid_runs/weights/{RUN_NAME}{os.path.splitext(BEST)[1]}")
result = {
    "run_name": RUN_NAME, "model_id": MODEL, "architecture": ARCH, "seed": SEED,
    "dataset": {"project": PROJECT, "version": VERSION, "n_test": SPLIT_IMAGES["test"], "classes": NAMES,
                "resplit": {"seed": SEED, "images": SPLIT_IMAGES, "manifest_md5": MANIFEST_MD5,
                            "boxes": {s: counts[s].to_dict() for s in SPLITS}}},
    "constants": {"imgsz": IMGSZ, "epochs": EPOCHS, "patience": PATIENCE, "batch": BATCH,
                  "threshold_mode": "best F1 on valid, sweep 0.05-0.95", "model_input": int(MODEL_INPUT)},
    "env": {"device_name": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
            "torch": torch.__version__, "library": f"{FAMILY} {LIB_VERSION}"},
    "leakage": {"test_in_train_frac": LEAK["test_in_train_frac"], "valid_in_train_frac": LEAK["valid_in_train_frac"]},
    "conf_threshold": CONF, "safety_classes": list(SAFETY_CLASSES), "safety_recall_test": SAFETY_RECALL,
    "params_m": PARAMS_M, "weights_mb": WEIGHTS_MB, "train_seconds": TRAIN_SECONDS, "latency": LAT,
    "valid": {"map50": val_map50, "map50_95": val_map, "threshold_sweep": SWEEP.round(4).to_dict("records")},
    "test": {"map50": map50, "map50_95": map5095, "precision": P, "recall": R, "per_class": per_class},
}
out = f"{RESULTS_DIR}/{RUN_NAME}.json"
json.dump(result, open(out, "w"), indent=2)
print("saved", out)
print("Download it from the Output tab, or attach this notebook's output to the comparison notebook.")

saved /kaggle/working/visionaid_runs/results/yolo11n_v4_seed0.json
Download it from the Output tab, or attach this notebook's output to the comparison notebook.
